### Data Profiling

In [3]:
import os
from pyspark.sql import SparkSession

os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 pyspark-shell'

AWS_ACCESS_KEY = "***********"
AWS_SECRET_KEY = "**********************"
BUCKET_NAME = "maintenance-bonn-2026" 

print("Initializing Fresh Spark Session with AWS Jars...")

Initializing Fresh Spark Session with AWS Jars...


In [4]:
spark = SparkSession.builder \
    .appName("S3_Direct_Read") \
    .getOrCreate()

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", AWS_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", AWS_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "s3.eu-north-1.amazonaws.com") 
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

print("Spark is connected to AWS!")

Spark is connected to AWS!


In [6]:
# Reading from S3
s3_file_path = f"s3a://{BUCKET_NAME}/bronze/ai4i2020_raw.csv"
print(f"Reading data directly from: {s3_file_path}")

df_s3 = spark.read.csv(s3_file_path, header=True, inferSchema=True)

print("\n--- First 5 Rows from S3 ---")
df_s3.show(5)

Reading data directly from: s3a://maintenance-bonn-2026/bronze/ai4i2020_raw.csv

--- First 5 Rows from S3 ---
+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|UDI|Product ID|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|  1|    M14860|   M|              298.1|                  308.6|                  1551|       42.8|              0|              0|  0|  0|  0|  0|  0|
|  2|    L47181|   L|              298.2|                  308.7|                  1408|       46.3|              3|              0|  0|  0|  0|  0|  0|
|  3|    L47182|   L|              298.1|                  308.5|                  1498|       49.4|              5|         

In [7]:
row_count = df_s3.count()
col_count = len(df_s3.columns)
print(f"\nTotal Rows: {row_count}")
print(f"Total Columns: {col_count}")


Total Rows: 10000
Total Columns: 14


### Data Cleaning

In [15]:
import pyspark.sql.functions as F
cleaned_df = df_s3

# Cleaning Column Names
for col_name in cleaned_df.columns:
    new_name = col_name.replace(" ", "_").replace("[", "").replace("]", "").lower()
    cleaned_df = cleaned_df.withColumnRenamed(col_name, new_name)

# Metadata
cleaned_df = cleaned_df.withColumn("processed_at", F.current_timestamp())

In [16]:
# Final Result
print("\n--- Cleaned Schema ---")
cleaned_df.printSchema()
print("\n--- First 5 Cleaned Rows ---")
cleaned_df.show(5)


--- Cleaned Schema ---
root
 |-- udi: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- air_temperature_k: double (nullable = true)
 |-- process_temperature_k: double (nullable = true)
 |-- rotational_speed_rpm: integer (nullable = true)
 |-- torque_nm: double (nullable = true)
 |-- tool_wear_min: integer (nullable = true)
 |-- machine_failure: integer (nullable = true)
 |-- twf: integer (nullable = true)
 |-- hdf: integer (nullable = true)
 |-- pwf: integer (nullable = true)
 |-- osf: integer (nullable = true)
 |-- rnf: integer (nullable = true)
 |-- processed_at: timestamp (nullable = false)


--- First 5 Cleaned Rows ---
+---+----------+----+-----------------+---------------------+--------------------+---------+-------------+---------------+---+---+---+---+---+--------------------+
|udi|product_id|type|air_temperature_k|process_temperature_k|rotational_speed_rpm|torque_nm|tool_wear_min|machine_failure|twf|hdf|pwf|osf|rnf|  

In [17]:
# Saving Cleaned Data to Parquet format 
print("Saving Cleaned Data directly to AWS S3 (Silver Layer)...")
s3_output_path = f"s3a://{BUCKET_NAME}/silver/ai4i2020_cleaned"
cleaned_df.write.mode("overwrite").parquet(s3_output_path)
print(f"Success! Silver Layer is fully processed and securely saved at: {s3_output_path}")

Saving Cleaned Data directly to AWS S3 (Silver Layer)...
Success! Silver Layer is fully processed and securely saved at: s3a://maintenance-bonn-2026/silver/ai4i2020_cleaned
